## Анализ внимания (Attention) с BertViz
Трансформер опирается на механизмы внимания, и разные головы могут фокусироваться на разных аспектах текста. Инструмент [BertViz](https://github.com/jessevig/bertviz) позволяет интерактивно визуализировать, как каждая голова модели распределяет внимание между токенами. Можно выбрать "head view" (отдельные головы слоя) или "model view" (обзор по всем слоям и головам).

Пример использования BertViz представлен в коде ниже. BertViz хорошо работает в ноутбуках и поддерживает большинство моделей Hugging Face. Для корректной работы при загрузке модели включите `output_attentions=True`.

In [2]:
from bertviz import head_view
from transformers import AutoTokenizer, AutoModel
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name, ignore_mismatches=True)
model = AutoModel.from_pretrained(model_name, output_attentions=True, ignore_mismatched_sizes=True)

sentence = "Иван Иванович Иванов работает в Газпроме."
inputs = tokenizer.encode(sentence, return_tensors='pt')
outputs = model(inputs)
attention = outputs.attentions  # список матриц внимания по слоям

# Покажем head-view для слоя 0, голова 0 (пример)
head_view(attention, tokenizer.convert_ids_to_tokens(inputs[0]), layer=0, heads=0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5892.83it/s]


<IPython.core.display.Javascript object>

## Задание 1
Проанализируйте, как модель «смотрит» на текст через механизм внимания. 

Допишите код, который будет токенизировать и возвращать attention-карты для разных слоёв и голов с помощью `head_view` из `BertViz`. 

Начните с промежуточных слоёв, например, Layer 2 (в интерфейсе `bertviz` это layer=2). Просмотрите несколько голов в этом слое: в средних слоях часто видно связывание частей имён.

Затем сравните с верхними слоями, там больше синтаксических и долгосрочных связей. Иногда лучше видно отделение служебных слов (предлогов) от имён — модель учится считать их менее информативными для смысла и фокусируется на опорных словах.
Используйте модель `cointegrated/rubert-tiny2` с `output_attentions=True`.

In [3]:
from transformers import AutoModel, AutoTokenizer
from bertviz import head_view
import torch
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()


# Модель для анализа attention (без головы классификации)
tokenizer = AutoTokenizer.from_pretrained('cointegrated/rubert-tiny2', ignore_mismatches=True)
attention_model = AutoModel.from_pretrained('cointegrated/rubert-tiny2', output_attentions=True, ignore_mismatched_sizes=True)

def analyze_attention(sentence, layer=0, heads=0):
    """Анализирует attention для заданного предложения"""
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    print(f"Анализируем предложение: {sentence}")
    print(f"Токены: {tokens}")
    
    with torch.no_grad():
        # Получите outputs
        outputs = attention_model(**inputs)
        # Получите attentions из outputs
        attentions = outputs.attentions
    
    print(f"Количество слоёв: {len(attentions)}")    
    # Визуализация конкретной головы
    head_view(attentions, tokens, layer=layer, heads=heads)

    
example_sentence = "Иван Петров работает в Google."
analyze_attention(example_sentence, layer=2, heads=0)
example_sentence = "Иван Иванович Иванов работает в Газпроме."
analyze_attention(example_sentence, layer=2, heads=0)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7137.36it/s]

Анализируем предложение: Иван Петров работает в Google.
Токены: ['[CLS]', 'Иван', 'Петров', 'работает', 'в', 'Google', '.', '[SEP]']
Количество слоёв: 3


<IPython.core.display.Javascript object>

Анализируем предложение: Иван Иванович Иванов работает в Газпроме.
Токены: ['[CLS]', 'Иван', 'Иванович', 'Иванов', 'работает', 'в', 'Газпром', '##е', '.', '[SEP]']
Количество слоёв: 3


<IPython.core.display.Javascript object>

## Подготовка данных и модели
### О датасете
В этом уроке мы будем использовать `FactRuEval-2016`. Это классический набор для русского NER и других задач информационного извлечения. Он содержит несколько дорожек оценки, разметку спанов, объектов и кореференции. 

Данные доступны в [репозитории](https://github.com/dialogue-evaluation/factRuEval-2016). Поэтому, вам следует склонировать себе этот репозиторий на ВМ.

Можно сказать, что данные сосредоточены в папке `devset` внутри репозитория. Сами файлы можно диференциировать в зависимости от расширения: \*.txt (тексты), \*.tokens (токенизация и позиции), \*.spans (спаны — границы сущностей), \*.objects (объекты и типы сущностей) и \*.coref (кореференция и идентификация). То есть в наборе уже есть разметка спанов по символам и разбиение на токены, — это помогает корректно выровнять метки по символам.

Заметим, что в репозитории также есть папка `testset` , в которой могут встречаться ошибки. 

## Задание 2
Реализуйте простой токенайзер по непробельным сегментам (\S+), который вернёт список токенов и соответствующие им [start,end) смещения. Этого достаточно для выравнивания по символам.  
Вам нужно:
- Получить переменную records список записей из датасета (обёрнутый list(...) от corus.load_factru(dir_path)).
- Реализовать функцию whitespace_tokenize_with_offsets(text), которая токенизирует входную строку по непробельным сегментам и возвращает кортеж (tokens, spans): tokens — список токенов (строк), spans — список пар (start, end), где start — индекс первого символа токена в исходном тексте, end — индекс после последнего символа.  
- Используйте регулярное выражение \S+ (последовательности непробельных символов). Это приемлемый и простой вариант для задач выравнивания по символам.


In [4]:
import subprocess

repo_url = "https://github.com/dialogue-evaluation/factRuEval-2016"
local_dir = "factRuEval-2016/"

try:
    result = subprocess.run(
        ["git", "clone", repo_url, local_dir],
        check=True,
        capture_output=True,
        text=True
    )
    print("Репозиторий успешно склонирован!")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("Ошибка при клонировании репозитория:")
    print(e.stderr)


Ошибка при клонировании репозитория:
fatal: destination path 'factRuEval-2016' already exists and is not an empty directory.



In [5]:
from corus import load_factru
import re

dir_path = "factRuEval-2016/" 

# Загрузите записи
records = list(load_factru(dir_path))
print("Загружено записей:", len(records))

# Реализуйте whitespace-tokenizer
def whitespace_tokenize_with_offsets(text: str):
    tokens = re.findall(r'\S+', text)
    spans = [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]
    return tokens, spans

example_sentence = "Иван Петров работает в Google."
whitespace_tokenize_with_offsets(example_sentence)

Загружено записей: 254


(['Иван', 'Петров', 'работает', 'в', 'Google.'],
 [(0, 4), (5, 11), (12, 20), (21, 22), (23, 30)])

In [6]:
unique_labels = set()
for record in records:
    for obj in record.objects:
        unique_labels.add(obj.type)
print("Уникальные метки в исходном датасете:\n", unique_labels)

Уникальные метки в исходном датасете:
 {'Facility', 'Location', 'Project', 'Org', 'Person', 'LocOrg'}


## Задание 3
Напишите функцию `map_object_type(obj_type)` для приведения разнообразных типов из FactRu к базовым NER-классам `PER, ORG, LOC, MISC`. Это поможет унифицировать метки в BIO-формате. На вход подаётся строка с исходным типом из разметки FactRu. Она может быть в разном регистре и содержать подтипы, например, person или LocOrg. Ваша функция должна определять, относится ли эта строка к сущностям PER, ORG, LOC и возвращать соответствующую метку, а в противном случае возвращать MISC.

In [7]:
def map_object_type(obj_type: str) -> str:
    """
    Сопоставляет строковое обозначение типа объекта (obj_type) к одной из базовых меток:
    {'PER', 'ORG', 'LOC', 'MISC'}.

    Вход:
        obj_type (str): строка с исходным типом сущности из разметки FactRu.
                        Может быть в разном регистре и содержать подтипы, например:
                        "person", "Organization", "DATE", "product" и т.д.

    Возвращает:
        str: одна из {'PER','ORG','LOC','MISC'}.

    Примеры:
        map_object_type("PER") -> "PER"
        map_object_type("person_name") -> "PER"
        map_object_type("organization") -> "ORG"
        map_object_type("GPE") -> "LOC"
        map_object_type("event") -> "MISC"
    """
    t = (obj_type or "").lower()
    if "person" in t or t in {"person", "name", "surname", "firstname", "patronymic"}:
        return "PER"
    if "org" in t or "organization" in t or "company" in t or "org_name" in t or "org_descr" in t:
        return "ORG"
    if "loc" in t or "location" in t or "geo" in t or "place" in t or "loc_name" in t:
        return "LOC"
    return "MISC" 

In [8]:
records[0].objects[0].spans[0]

FactruSpan(
    id='25856',
    type='org_name',
    start=160,
    stop=186
)

## Задание 4
Для каждого документа мы пройдём по всем объектам и их спанам и пометим соответствующие токены как `B-TYPE` / `I-TYPE`. 

Если токен уже имеет метку (например, два спана перекрываются), оставляем первую метку — это простой и безопасный подход. Если не контролировать перезапись, данные станут непоследовательными. Плохая разметка — плохие признаки для обучения. 

Если span не пересекается ни с одним токеном — пропускаем его.

Модель обучается предсказывать метку на каждом токене. 

Что нужно получить: список examples, где каждый элемент — dict с ключами `id, text, tokens, tags` (tags — список строк типа B-PER/I-ORG/O).

In [9]:
examples = []

for rec in records:
    # 1) Возьмём текст
    text = rec.text

    # 2) Токенизация + спаны токенов (offsets)
    tokens, token_spans = whitespace_tokenize_with_offsets(text)

    # 3) Изначально все метки = "O"
    token_labels = ["O"] * len(tokens)

    # 4) Пройдём по объектам и их спанам
    for obj in rec.objects:
        # Приводим тип сущности к базовому (PER/ORG/LOC/...)
        base_type = map_object_type(obj.type)

        for span in obj.spans:
            span_start = span.start
            span_end = span.stop

            # 5) Найдём индексы токенов, пересекающихся со span
            overlapping_idxs = []
            for i, (t_start, t_end) in enumerate(token_spans):
                # Условие пересечения полуинтервалов [t_start, t_end) и [span_start, span_end)
                if t_start <= span_end and t_end >= span_start:
                    overlapping_idxs.append(i)

            # 6) Если span не пересёк ни один токен — пропускаем
            if not overlapping_idxs:
                continue

            # 7) Проставим B-/I- метки по всем пересекающимся токенам
            for j, tok_idx in enumerate(overlapping_idxs):
                # Если токен уже размечен (перекрытие) — оставляем первую метку
                if token_labels[tok_idx] != "O":
                    continue

                prefix = "B" if j == 0 else "I" # "B" для первого, иначе "I"
                token_labels[tok_idx] = f"{prefix}-{base_type}"

    # 8) Сохраним пример
    examples.append({
        "id": rec.id,
        "text": text,
        "tokens": tokens,
        "tags": token_labels,
    })

print(f"Примеры собраны: {len(examples)}")
print("Пример tokens:", examples[2]["tokens"][:20])
print("Пример tags:  ", examples[2]["tags"][:20])

Примеры собраны: 254
Пример tokens: ['Около', '30', 'рабочих', 'на', 'заводе', 'по', 'производству', 'ветрогенераторов', 'Vestas,', 'расположенном', 'в', 'Ньюпорте,', 'столицы', 'Острова', 'Уайт,', 'Англия', 'захватили', 'свой', 'завод', 'в']
Пример tags:   ['O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'B-ORG', 'O', 'O', 'B-LOC', 'O', 'B-LOC', 'B-LOC', 'B-LOC', 'O', 'O', 'O', 'O']


В FactRu есть вложенные и сложные объекты. Наш подход «оставить первую метку» подходит для эксперимента, но для продашна можно применять приоритет (по длине спана, по типу сущности) или другие правила.

Код ниже собирает детерминированный label_list (включая O), строит label2id/id2label и переводит examples["tags"] в числовые id, создаёт Dataset и развивает на train/test (например, 90/10). Используйте datasets.DatasetDict в следующем задании.

In [10]:
from datasets import Dataset, DatasetDict

unique_labels = set()
for ex in examples:
    unique_labels.update(ex["tags"])
unique_labels.add("O")
label_list = sorted(unique_labels)
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}

for ex in examples:
    ex["tags"] = [label2id[t] for t in ex["tags"]]

full_ds = Dataset.from_list(examples)
split = full_ds.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({"train": split["train"], "test": split["test"]})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'tags'],
        num_rows: 228
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'tags'],
        num_rows: 26
    })
})


## Задание 5
У нас уже есть word-level токены и numeric tags. 

Допишите функцию — отправьте tokens в BERT-токенизатор с параметром инициализации (`is_split_into_words=True`) и выровняйте метки под субтокены. 

Как выравнивать метки. Первый субтокен слова получает метку, остальные получают -100. Для специальных токенов и PAD также ставим -100. 

Почему так. Модель работает на уровне subword. При этом мы хотим, чтобы loss учитывал только первые subword-части слов, а не все фрагменты — иначе одно длинное имя, разбитое на 4 части, даст 4 сигнала и испортит статистику. Такой подход к выравниванию меток — стандартная практика для token-classification, она совместима с CrossEntropyLoss. 

Если нужно, подберите max_length. Учитывайте, что при слишком маленьком max_length длинные предложения обрезаются, а при слишком большом — модель обучается медленнее.


In [16]:
from transformers import AutoTokenizer
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", is_split_into_words=True, ignore_mismatches=True)

def tokenize_and_align_labels(examples_batch):
    tokenized = tokenizer(
        examples_batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )
    labels = []
    for i, word_labels in enumerate(examples_batch["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            # Условие: если word_idx == None, то это padding/special token
            # Иначе, если word_idx != prev_word_idx, то это начало нового слова
            # Ваш код здесь
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)
            prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized


# 8. Применяем токенизацию к датасету
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["text", "tokens", "tags", "id"]
)

print(tokenized_dataset)
# Проверка: показываем один пример из train
print(tokenized_dataset["train"][0])

Map: 100%|██████████| 26/26 [00:00<00:00, 661.14 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 228
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 26
    })
})
{'input_ids': [101, 1194, 29750, 15290, 18947, 23742, 17432, 18947, 1193, 29740, 25529, 10325, 18947, 10325, 29436, 1196, 15290, 29748, 29436, 1182, 1196, 16856, 29113, 25529, 15290, 1194, 15290, 16856, 15290, 29741, 19259, 14150, 16856, 19259, 1182, 1187, 10260, 17432, 25529, 29436, 15290, 18947, 15414, 1189, 18947, 29742, 16856, 1183, 19259, 14150, 16856, 10325, 22919, 29747, 17432, 1010, 1202, 22919, 14150, 1184, 10325, 10260, 29436, 14150, 29741, 1192, 15290, 1198, 29742, 10260, 29436, 29747, 17432, 1188, 29744, 1011, 1187, 10260, 1197, 14150, 29741, 14150, 1010, 1202, 22919, 14150, 1196, 15290, 29748, 29436, 1188, 29744, 19865, 29752, 10260, 29436, 23742, 18947, 14150, 1077, 1198, 29745, 29113, 29753, 29436, 1529

## Задание 6
Подготовьте `DataCollatorForTokenClassification` и `DataLoader`. Нужно получить батчи с объединёнными тензорами. Создайте `DataLoader` с `batch_size=16` и `shuffle=True`.

Задание небольшое. Выполняйте его на ВМ, полученной в прошлом уроке. Затем сверьтесь с авторским решением. 

In [12]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

# Предполагается, что tokenized_dataset уже создан (после dataset.map(tokenize_and_align_labels))
# Ваш код здесь: создайте data_collator, train_dataloader
data_collator = DataCollatorForTokenClassification(tokenizer)
train_dataloader = DataLoader(tokenized_dataset["train"], batch_size=16, shuffle=True, collate_fn=data_collator)

print("Готово. Примеры для обучения:", len(tokenized_dataset["train"]))

Готово. Примеры для обучения: 228


In [17]:
# Функция, выравнивающая предсказания модели и реальные метки (на уровне tokenized_dataset)
def get_flat_labels_and_preds_from_model(tokenized_split, model, device, max_samples=None):
    """
    tokenized_split: dataset split (list-like of examples with keys 'input_ids','attention_mask','labels')
    Возвращает flat lists: y_true (ints), y_pred (ints)
    """
    y_true = []
    y_pred = []
    for i, ex in enumerate(tokenized_split):
        if max_samples is not None and i >= max_samples:
            break

        # Превращаем в тензоры (batch size = 1)
        input_ids = torch.tensor([ex["input_ids"]], dtype=torch.long).to(device)
        attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # shape (1, seq_len, num_labels)
            preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()  # list длины seq_len

        # Истинные метки (включая -100 для пэддинга/ignored)
        true_labels = ex["labels"]  # список длиной seq_len; элементы -100 или id

        # Фильтруем позиции, где true != -100
        filtered_true = []
        filtered_pred = []
        for p, t in zip(preds, true_labels):
            if t == -100:
                continue
            filtered_true.append(int(t))
            filtered_pred.append(int(p))

        # Обрежем на минимальную длину (на случай рассинхронизации)
        minlen = min(len(filtered_true), len(filtered_pred))
        if minlen == 0:
            continue
        y_true.extend(filtered_true[:minlen])
        y_pred.extend(filtered_pred[:minlen])

    return y_true, y_pred

## Задание 7
Получите baseline-оценку качества предобученной модели на tokenized test-сете с использованием `get_flat_labels_and_preds_from_model`. Вычислите token-level метрики: `precision / recall / F1 (average="macro")`.

Что нужно сделать по шагам:
1. Убедиться, что `tokenized_dataset`, `label2id`, `id2label`, `tokenizer` уже подготовлены.
1. Подгрузить предобученную модель `AutoModelForTokenClassification` (в режиме eval). Пример переменной `model_checkpoint` задан заранее.
1. Вызвать `get_flat_labels_and_preds_from_model(tokenized_dataset["test"], model, device, max_samples=200)` и получить `y_true` и `y_pred`.
1. Посчитать token-level метрики с помощью sklearn.metrics и вывести их с `average="macro"`.

In [19]:
from transformers import AutoModelForTokenClassification
from sklearn.metrics import precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем модель 
model = AutoModelForTokenClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

model.to(device)
model.eval()
# Получите token-level предсказания и истинные метки с помощью готовой функции
# (max_samples можно уменьшить/увеличить по ресурсам)
y_true, y_pred = get_flat_labels_and_preds_from_model(tokenized_dataset["test"], model, device, max_samples=200)

baseline_metrics = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)

# Посчитайте метрики
print("Samples used (token-level):", len(y_true))
print("Precision:", baseline_metrics[0])
print("Recall:   ", baseline_metrics[1])
print("F1:       ", baseline_metrics[2])

# Сохраните или запишите полученные численные значения как baseline.

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 10472.42it/s]

Samples used (token-level): 546
Precision: 0.1324938800113012
Recall:    0.08965568372023337
F1:        0.0755229650264407


## Задание 8
Дообучите модель на `tokenized_dataset["train"]` с использованием `PyTorch` (`DataLoader + DataCollatorForTokenClassification`). Затем проведите ту же token-level оценку на test-сете. Сравните метрики до и после дообучения.

Что нужно сделать:
1. Запустить цикл обучения (перезапись `optimizer.step()`, `zero_grad` и т. д.).
1. Вывести сравнение: `baseline` vs `fine-tuned` и `Delta F1`.

In [22]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

# Параметры 
num_epochs = 20         # уменьшите при необходимости
learning_rate = 5e-5


# 2) Оптимизатор
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 3) Loop обучения
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        # Перенести batch на device, вычислить loss, backward, step, zero_grad
        # Ваш код здесь
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch+1} avg loss: {total_loss/len(train_dataloader):.4f}")

# 4) Оценка после fine-tune
y_true_ft, y_pred_ft = get_flat_labels_and_preds_from_model(
    tokenized_dataset["test"], model, device, max_samples=200
)

print("After fine-tuning:")
print("Precision:", precision_score(y_true_ft, y_pred_ft, average="macro", zero_division=0))
print("Recall:   ", recall_score (y_true_ft, y_pred_ft, average="macro", zero_division=0))
print("F1:       ", f1_score   (y_true_ft, y_pred_ft, average="macro", zero_division=0))

# 5) Сравнение с baseline (предполагается, что baseline метрики сохранены)
# Ваш код: загрузите baseline F1 и выведите delta
# Ваш код здесь
baseline_f1 = baseline_metrics[2]
delta = f1_score(y_true_ft, y_pred_ft, average="macro", zero_division=0) - baseline_f1
print("Delta:    ", delta)

Epoch 1: 100%|██████████| 15/15 [00:04<00:00,  3.15it/s]


Epoch 1 avg loss: 0.7166


Epoch 2: 100%|██████████| 15/15 [00:04<00:00,  3.13it/s]


Epoch 2 avg loss: 0.6871


Epoch 3: 100%|██████████| 15/15 [00:04<00:00,  3.21it/s]


...

Epoch 18 avg loss: 0.4040


Epoch 19: 100%|██████████| 15/15 [00:04<00:00,  3.64it/s]


Epoch 19 avg loss: 0.3796


Epoch 20: 100%|██████████| 15/15 [00:03<00:00,  3.82it/s]


Epoch 20 avg loss: 0.3862
After fine-tuning:
Precision: 0.2583517427589593
Recall:    0.20358000519977645
F1:        0.21391670527469378
Delta:     0.13839374024825307
